# 实验五：Jetson Nano 实时四色目标识别（稳定双画面版）

依据现有 `week5_实时双画面.ipynb` 修改，参考 `2.1.在OpenCV 中调用摄像头.ipynb` 的 `ipywidgets.Image → cv2.imencode → .value`。仅显示**原始画面**与**红黄绿蓝过滤画面**，不会使用 `cv2.imshow()`。保留 HSV 标定、截图及不同光照记录。

**依次运行 1–5 节。** 采集默认持续 30 秒；若设为 0，使用 Jupyter 的 Interrupt 停止。硬件测试需在 Jetson Nano 实机进行。

## 1. 环境与配置（统一摄像头变量）

In [1]:
import os, re, glob, json, csv, time, signal, subprocess
from pathlib import Path
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# 摄像头统一配置；不在其他代码单元设置编号
CAMERA_DEVICE = None
AUTO_RELEASE_OTHER_CAMERA_PROCESSES = False  # 默认安全：只报告占用，不关闭进程
PREFERRED_DEVICE = None  # 如需指定可设 '/dev/video1'；默认自动探测
FRAME_WIDTH, FRAME_HEIGHT = 640, 480  # 只作为偏好，必须按设备实际能力协商
CAMERA_FPS = 15
FPS_LIMIT = 8
JPEG_QUALITY = 70
RUN_SECONDS = 30  # 设为 0 后持续运行，需手动 Interrupt Kernel
MAX_READ_FAILURES = 3
OUTPUT_DIR = Path('week5_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HSV_RANGES = {
    'red':    [((0, 95, 65), (10, 255, 255)), ((170, 95, 65), (179, 255, 255))],
    'yellow': [((18, 85, 65), (38, 255, 255))],
    'green':  [((39, 65, 50), (85, 255, 255))],
    'blue':   [((90, 75, 50), (135, 255, 255))],
}

# 保留内核中的单一摄像头句柄；重跑本节不抹除已有句柄，下一节统一释放
if '_camera_state' not in globals():
    _camera_state = {'cap': None, 'active': False, 'device': None}
if 'last_original' not in globals(): last_original = None
if 'last_filtered' not in globals(): last_filtered = None
print('主机:', os.uname().nodename, '| OpenCV:', cv2.__version__, '| PID:', os.getpid())

主机: nano | OpenCV: 4.5.1 | PID: 21491


## 2. 摄像头发现、占用检查和安全释放

自动枚举 `/dev/video*`，优先利用 `v4l2-ctl` 查询视频采集格式（工具不存在时继续尝试 OpenCV）。**探测即真实试读一帧**，遇到读取失败会释放句柄再尝试其他候选设备/配置。

占用检查默认只报告。启用自动清理也**仅考虑同一用户的其他 Jupyter 内核**，对于普通 Python、系统服务或当前内核均不自动结束；在无人监管情况下不应执行广泛的 kill。

In [2]:
def release_camera():
    """释放当前内核保管的唯一摄像头句柄。"""
    state = _camera_state
    cap = state.get('cap')
    state['cap'] = None
    state['active'] = False
    state['device'] = None
    if cap is not None:
        try: cap.release()
        except Exception as exc: print('释放摄像头时出现异常:', exc)


def _camera_pids(device):
    try:
        r = subprocess.run(['fuser', device], capture_output=True, text=True, timeout=3)
        return [int(x) for x in re.findall(r'\b\d+\b', r.stdout)]
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print('提示：fuser 未安装或超时，无法检查设备占用')
        return []


def inspect_camera_users(device):
    current = os.getpid()
    found = []
    for pid in _camera_pids(device):
        try:
            raw = Path(f'/proc/{pid}/cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace').strip()
            status = Path(f'/proc/{pid}/status').read_text()
            uid = int(re.search(r'^Uid:\s+(\d+)', status, re.M).group(1))
        except (FileNotFoundError, ProcessLookupError, PermissionError, AttributeError):
            raw, uid = '(无权读取或进程已经结束)', None
        info = {'pid': pid, 'command': raw, 'uid': uid, 'current': pid == current}
        found.append(info)
        print(f"设备 {device}：PID={pid} UID={uid} 当前内核={pid == current} 命令={raw[:180]}")
    if not found: print(f'{device}: 未发现打开设备的进程')
    return found


def optionally_release_other_kernels(device):
    found = inspect_camera_users(device)
    others = [p for p in found if not p['current']]
    if not AUTO_RELEASE_OTHER_CAMERA_PROCESSES or not others:
        if others: print('默认不自动终止进程。请在 Jupyter Running 面板关闭闲置内核。')
        return
    for p in others:
        # 为避免误杀，自动操作仅限当前用户、命令行可确认的其他 ipykernel
        cmd = p['command']
        if p['uid'] != os.getuid() or 'ipykernel_launcher' not in cmd or 'jupyter-lab' in cmd:
            print(f"跳过 PID={p['pid']}：不是可确认的本用户 Jupyter 内核")
            continue
        if p['pid'] not in _camera_pids(device):
            continue
        try:
            os.kill(p['pid'], signal.SIGTERM)
            print(f"已向其他 Jupyter 内核 PID={p['pid']} 发送 SIGTERM")
        except (PermissionError, ProcessLookupError) as exc:
            print('无法终止进程:', exc)
    time.sleep(0.7)
    inspect_camera_users(device)


def candidate_devices():
    found = sorted(glob.glob('/dev/video*'), key=lambda p: int(re.search(r'\d+$', p).group(0)))
    if PREFERRED_DEVICE and PREFERRED_DEVICE in found:
        return [PREFERRED_DEVICE] + [p for p in found if p != PREFERRED_DEVICE]
    return found


def _supported_profiles(device):
    # 格式的可选信息：不安装 v4l-utils 也可运行
    profiles = []
    try:
        r = subprocess.run(['v4l2-ctl', '-d', device, '--list-formats-ext'],
                           capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            formats = re.findall(r"'((?:MJPG|YUYV))'", r.stdout)
            for fmt in dict.fromkeys(formats):
                profiles.append((fmt, FRAME_WIDTH, FRAME_HEIGHT, CAMERA_FPS))
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    # 默认配置永远作为回退；即使无指定格式也尝试采集
    profiles.append((None, None, None, None))
    return profiles


def _try_open(device, profile):
    fmt, width, height, fps = profile
    cap = cv2.VideoCapture(device, cv2.CAP_V4L2)
    if not cap.isOpened():
        cap.release()
        return None, None
    try:
        if fmt:
            cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*fmt))
        if width: cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        if height: cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        if fps: cap.set(cv2.CAP_PROP_FPS, fps)
        # 读取超时可能较长，限制重试次数，不在超时时调用 cvtColor
        for _ in range(2):
            ret, frame = cap.read()
            if ret and frame is not None and frame.size:
                return cap, frame
        cap.release()
        return None, None
    except Exception:
        cap.release()
        raise


def open_camera():
    global CAMERA_DEVICE
    if _camera_state['active']:
        raise RuntimeError('已有实时采集正在运行；请先中断该单元，避免并行读摄像头')
    release_camera()
    devices = candidate_devices()
    if not devices:
        raise RuntimeError('没有找到 /dev/video*；请检查摄像头连接与 USB 枚举')
    print('视频设备候选:', devices)
    for device in devices:
        optionally_release_other_kernels(device)
        if any(p['pid'] != os.getpid() for p in inspect_camera_users(device)):
            print('该设备可能被其他进程占用，跳过:', device)
            continue
        for profile in _supported_profiles(device):
            print('尝试:', device, '配置:', profile)
            cap, frame = _try_open(device, profile)
            if cap is not None:
                CAMERA_DEVICE = device
                _camera_state.update(cap=cap, device=device)
                print('采集成功:', device, '| 帧尺寸:', frame.shape)
                return cap, frame
    raise RuntimeError('没有找到能够成功读取帧的设备；请检查 USB 供电/连接及 v4l2-ctl --list-formats-ext')

print('设备:', candidate_devices())

设备: ['/dev/video1']


## 3. 单帧采集验证（测试后立即释放，不产生额外显示画面）

In [3]:
try:
    camera, sample_frame = open_camera()
    print('单帧测试成功:', CAMERA_DEVICE, 'shape:', sample_frame.shape)
except RuntimeError as exc:
    print('单帧测试失败:', exc)
finally:
    release_camera()

视频设备候选: ['/dev/video1']
/dev/video1: 未发现打开设备的进程
/dev/video1: 未发现打开设备的进程
尝试: /dev/video1 配置: ('YUYV', 640, 480, 15)


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.


尝试: /dev/video1 配置: ('MJPG', 640, 480, 15)


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.


尝试: /dev/video1 配置: (None, None, None, None)


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.


单帧测试失败: 没有找到能够成功读取帧的设备；请检查 USB 供电/连接及 v4l2-ctl --list-formats-ext


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.


## 4. HSV 过滤及 JPEG 工具函数（无需摄像头即可测试）

In [4]:
def filter_four_colors(frame_bgr, ranges=None):
    if frame_bgr is None or frame_bgr.size == 0:
        raise ValueError('输入图像为空，不能执行 HSV 转换')
    ranges = HSV_RANGES if ranges is None else ranges
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    merged = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for intervals in ranges.values():
        for lower, upper in intervals:
            mask = cv2.inRange(hsv, np.asarray(lower, dtype=np.uint8),
                              np.asarray(upper, dtype=np.uint8))
            merged = cv2.bitwise_or(merged, mask)
    filtered = cv2.bitwise_and(frame_bgr, frame_bgr, mask=merged)
    return filtered, merged


def bgr8_to_jpeg(frame, quality=JPEG_QUALITY):
    ok, encoded = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok: raise RuntimeError('JPEG 编码失败')
    return encoded.tobytes()

print('HSV 转换及 JPEG 工具已就绪')

HSV 转换及 JPEG 工具已就绪


## 5. 双画面实时采集

**此单元唯一负责实时画面**。运行时展示两个 `ipywidgets.Image`：左原图、右四色过滤图。两者来自同一帧；不会打开其他图像窗口。

`RUN_SECONDS=30` 默认自动停止，设 `0` 可持续运行，使用 Jupyter Interrupt 停止。不要在其他 Notebook 中同时采集。

In [ ]:
original_widget = widgets.Image(format='jpeg', width=480, height=360)
filtered_widget = widgets.Image(format='jpeg', width=480, height=360)
display(widgets.HBox([
    widgets.VBox([widgets.HTML('<b>原始摄像头画面</b>'), original_widget]),
    widgets.VBox([widgets.HTML('<b>红黄绿蓝过滤画面</b>'), filtered_widget]),
]))

# 每次重跑此单元重置本轮状态
last_original, last_filtered = None, None
camera = None
try:
    camera, first_frame = open_camera()
    _camera_state['active'] = True
    start = time.monotonic()
    failures = 0
    pending_frame = first_frame
    while RUN_SECONDS == 0 or time.monotonic() - start < RUN_SECONDS:
        tick = time.monotonic()
        if pending_frame is not None:
            frame = pending_frame
            pending_frame = None
            ret = True
        else:
            ret, frame = camera.read()
        if not ret or frame is None or frame.size == 0:
            failures += 1
            if failures >= MAX_READ_FAILURES:
                print('连续读取失败；停止采集。请检查 select() timeout、USB 连接及摄像头格式。')
                break
            time.sleep(0.2)
            continue
        failures = 0
        filtered, mask = filter_four_colors(frame)
        original_widget.value = bgr8_to_jpeg(frame)
        filtered_widget.value = bgr8_to_jpeg(filtered)
        last_original, last_filtered = frame.copy(), filtered.copy()
        time.sleep(max(0, 1.0/max(1, FPS_LIMIT) - (time.monotonic()-tick)))
except KeyboardInterrupt:
    print('用户中断采集')
except RuntimeError as exc:
    print('采集未启动或已停止:', exc)
finally:
    release_camera()
    print('采集循环退出，摄像头已释放')

视频设备候选: ['/dev/video1']
/dev/video1: 未发现打开设备的进程
/dev/video1: 未发现打开设备的进程
尝试: /dev/video1 配置: ('YUYV', 640, 480, 15)


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.


尝试: /dev/video1 配置: ('MJPG', 640, 480, 15)


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video1): select() timeout.


尝试: /dev/video1 配置: (None, None, None, None)


## 6. 保存原图、过滤图及 HSV 参数

In [ ]:
if last_original is None or last_filtered is None:
    print('本轮没有成功采集帧；请先运行第5节。')
else:
    stamp = time.strftime('%Y%m%d_%H%M%S')
    a = OUTPUT_DIR / f'{stamp}_original.jpg'
    b = OUTPUT_DIR / f'{stamp}_filtered.jpg'
    ok1 = cv2.imwrite(str(a), last_original)
    ok2 = cv2.imwrite(str(b), last_filtered)
    if not (ok1 and ok2):
        raise RuntimeError('截图保存失败，请检查输出目录写入权限')
    path = OUTPUT_DIR / f'{stamp}_hsv.json'
    with path.open('w', encoding='utf-8') as f:
        json.dump(HSV_RANGES, f, ensure_ascii=False, indent=2)
    print('已保存：', a, b, path, sep='\n')

## 7. 不同光照实验记录

分别在正常、暗光、强光条件下采集截图，人工判断识别情况。不会预填或编造实验结果。调节第1节 `HSV_RANGES` 后，重新执行第5节即可观察参数效果。

In [ ]:
LIGHTING = 'normal'  # normal / dim / bright
TEST_COLOR = 'red'   # red / yellow / green / blue
RESULT = ''          # 正确 / 漏检 / 误检 / 部分检出
NOTES = ''           # 例如 阴影导致V值偏低

if not RESULT.strip():
    print('尚未填写 RESULT，未写入实验记录')
else:
    record = {
        'time': time.strftime('%Y-%m-%d %H:%M:%S'),
        'lighting': LIGHTING,
        'target_color': TEST_COLOR,
        'result': RESULT,
        'notes': NOTES,
        'camera_device': CAMERA_DEVICE,
        'hsv_ranges': json.dumps(HSV_RANGES, ensure_ascii=False)
    }
    csv_path = OUTPUT_DIR / 'lighting_tests.csv'
    new = not csv_path.exists()
    with csv_path.open('a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(record))
        if new: writer.writeheader()
        writer.writerow(record)
    print('已记录:', csv_path)

## 使用及故障处理

1. 若摄像头断开后从 `/dev/video0` 变为 `/dev/video1`，重新运行**第2节**和**第3节**自动发现设备，无需修改 `CAMERA_INDEX`。
2. 若看到占用 PID：先到 JupyterLab **Running** 面板关闭旧内核。自动结束其他内核功能默认关闭。
3. 若出现 `select() timeout`：说明视频流未提供帧，检查 `v4l2-ctl -d /dev/videoN --list-formats-ext`、USB 接口及供电；Notebook 无法代替硬件排障。
4. 若内核被外部 SIGKILL、断电或原生库崩溃，Python `finally` 无法保证执行。重开 Notebook 前检查 `fuser`。
5. 在采集循环执行时不要重跑初始化单元；使用 Jupyter Interrupt 停止后，必要时单独执行 `release_camera()`。
6. `HSV_RANGES` 为初始值，须根据不同光照进行实测标定。